In [1]:
import sys
import os 
import numpy as np
import pandas as pd
import re
from pulp import *

sys.path.append(os.path.abspath("../../")) ; from optimizer import variables
sys.path.insert(0, os.path.abspath("../../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *
from functools import partial
from tqdm import tqdm

pd.set_option('display.max_rows', None) ; pd.set_option('display.max_columns', None)

### Post optimization calculation logic

In [2]:
# Called for every window
def post_process_window_logic(self):
    
    def compute_interval_assumed_revenue():
    
        interval_assumed_revenue = [0.0] * len(self.dataset_window_subset)
        interval_demand_charges = [0.0] * len(self.dataset_window_subset)
        stored_charge = [value(v) for v in self.stored_charge]
        date = [v for v in self.dataset_window_subset['Date']]

        for component_name, interval_exprs in self.objective_component_map.items():
            for i, expr in interval_exprs.items():
                if i is not None and 0 <= i < len(self.dataset_window_subset):

                    interval_assumed_revenue[i] += value(expr)

                    if component_name == "monthly_import_export_penalties":
                        interval_demand_charges[i] += value(expr)
        
        # interval_assumed_revenue = interval_assumed_revenue + np.array(window_demand_charges)
                  
        return interval_assumed_revenue, interval_demand_charges, stored_charge, date
    

    def compute_interval_actual_revenue(window_demand_charges):

        n = len(self.dataset_window_subset)
        actual_price = self.dataset_window_subset["Actual_price"].to_numpy()
        lgc = self.dataset_window_subset["LGC"].to_numpy()
        import_charge = self.dataset_window_subset["Import_charge"].to_numpy()
        hte2grid = self.dataset_window_subset["hte2grid"].to_numpy()

        pv2grid_v = np.array([value(v) or 0.0 for v in self.pv2grid[:n]])
        dischargeE_v = np.array([value(v) or 0.0 for v in self.dischargeE[:n]])
        grid2battery_v = np.array([value(v) or 0.0 for v in self.grid2battery[:n]])
        dischargeE_eff = dischargeE_v * hte2grid

        revenue_expression = (
            (pv2grid_v + dischargeE_eff) * actual_price
            + (pv2grid_v + dischargeE_eff - grid2battery_v * variables.grid_import_penalty) * lgc
            - grid2battery_v * (actual_price + import_charge)
        )

        return revenue_expression + np.array(window_demand_charges)
    
    interval_assumed_revenue, interval_demand_charges, interval_stored_charge, interval_date = compute_interval_assumed_revenue()
    interval_actual_revenue = compute_interval_actual_revenue(interval_demand_charges)
    

    optimization_interval_results = pd.DataFrame({
            "interval_date":interval_date,
            "interval_stored_charge": interval_stored_charge,
            "interval_assumed_revenue": interval_assumed_revenue,
            "interval_actual_revenue": interval_actual_revenue,
        })
    
    
    return optimization_interval_results

### Export logic

In [3]:
# Only called once at the end
def export_outut_to_csv(retained_outputs, price_col_indexer, bess_duration_h):

    retained_outputs = retained_outputs.copy()
    retained_outputs["interval_date"] = pd.to_datetime(retained_outputs["interval_date"])

    output = {
        "price_col_indexer": price_col_indexer,
        "bess_duration_h": bess_duration_h,
    }

    annual_revenue = (
        retained_outputs.groupby(retained_outputs["interval_date"].dt.year)[
            ["interval_assumed_revenue", "interval_actual_revenue"]
        ]
        .sum()
        .round(0)
    )

    for year, row in annual_revenue.iterrows():
        output[f"assumed_revenue_{year}"] = row["interval_assumed_revenue"]
        output[f"actual_revenue_{year}"] = row["interval_actual_revenue"]

    return pd.DataFrame([output])


## Constraints:

#### Ballance PV output

In [4]:
def _interval_constraint_A(Lp, gen, pv2grid, pv2battery, curtailment, i):
    
    Lp += pv2grid + pv2battery + curtailment == gen, f"Ballance_PV_output_{i}"
    
    return Lp

#### Ballance BESS charge

In [5]:
def _interval_constraint_B(Lp, pv2battery_eff, grid2battery_eff, dischargeE, previous_interval_mwh, stored_charge,i):
   
    Lp += previous_interval_mwh + (pv2battery_eff + grid2battery_eff) - dischargeE == stored_charge, f"Ballance_BESS_charge_{i}"
    
    return Lp

#### Restrict BESS by BESS capacity

In [6]:
def _interval_constraint_C(Lp, pv2battery_eff, grid2battery_eff, dischargeE_eff, bess_energy_limit_per_interval_mw,i):

    Lp += pv2battery_eff + grid2battery_eff + dischargeE_eff <= bess_energy_limit_per_interval_mw, f"Restrict_BESS_by_BESS_capacity_{i}"

    return Lp

#### Restrict BESS by BESS physical limits

In [7]:
def _interval_constraint_D(Lp, stored_charge, bess_max_eff, chargeE, dischargeE, gen_exportable_now, pv2battery, gen_exportable,i):

    Lp += stored_charge <= bess_max_eff, f"Restrict_stored_charge_by_physcial_limit_{i}"

    Lp += chargeE <= bess_max_eff, f"Restrict_chargeE_by_physcial_limit_{i}"

    Lp += dischargeE <= bess_max_eff, f"Restrict_dischargeE_by_physcial_limit_{i}"

    # if gen_exportable_now == True:
    #     Lp += pv2battery <= gen_exportable
            
    return Lp

#### Restrict grid by grid capacity 

In [8]:
def _interval_constraint_E(Lp, pv2grid, dischargeE_eff, project_poi_export_limit_mwh_per_interval, grid2battery, project_poi_import_limit_mwh_per_interval,i):

  
    Lp += pv2grid + dischargeE_eff + grid2battery <= project_poi_export_limit_mwh_per_interval, f"Restrict_grid_by_grid_capacity_{i}"

    # Lp += grid2battery <= project_poi_import_limit_mwh_per_interval
    
    return Lp

#### Restrict BESS by signal mutuality

In [9]:
def _interval_constraint_F(Lp, charge_signal, discharge_signal, chargeE, dischargeE,i):

    if charge_signal == True:
        Lp += dischargeE == 0, f"Restrict_BESS_by_dischargeE_mutuality_{i}"
    
    if discharge_signal == True:
        Lp += chargeE == 0, f"Restrict_BESS_by_chargeE_mutuality_{i}"

    return Lp


#### Restrict grid importing during sun soaker times

In [10]:
def _interval_constraint_G(Lp, sun_soaker, grid2battery,i):
    
    if sun_soaker == True:
        Lp += grid2battery == 0,f"Restrict_importing_by_sunsoaker_{i}"
    
    return Lp

#### Restrict imports and exports to be equal or less than max monthly values

In [11]:
def _interval_constraint_H(Lp, tou_label, grid2battery, imports_max_peak, imports_max_shoulder, imports_max_offpeak, exports_max_sunsoaker, dischargeE_eff, pv2grid,i): 

        if tou_label == "Peak":
            Lp += grid2battery <= imports_max_peak, f"Restrict_onpeak_grid_imports_by_max_{i}"

        elif tou_label == "Shoulder":
            Lp += grid2battery <= imports_max_shoulder, f"Restrict_shoulder_grid_imports_by_max_{i}"

        elif tou_label == "Off-peak":
            Lp += grid2battery <= imports_max_offpeak, f"Restrict_peak_grid_imports_by_max_{i}"

        elif tou_label == "Sun Soaker":
            Lp += pv2grid + dischargeE_eff <= exports_max_sunsoaker, f"Restrict_sunsoaker_grid_exports_by_max_{i}"

        return Lp


#### Restrict BESS by cycle limit

In [12]:
def _window_constraint_A(Lp, window_cycle_budget_mwh, total_window_discharge):

    Lp += total_window_discharge <= window_cycle_budget_mwh, f"Restrict_BESS_by_cycle_limit"

    return Lp

## Panalties:

#### Penalise revenue by, price of, remaining BESS headroom, upon charge signal

In [13]:
def _interval_penalty_A(bess_capacity_remaining, charge_signal, trade_signal_penalty):
    
    if charge_signal:
        charge_signal_penalty = bess_capacity_remaining * trade_signal_penalty
        
        return charge_signal_penalty


#### Penalise revenue by, price of, remaining BESS stored charge, upon discharge signal

In [14]:
def _interval_penalty_B(discharge_signal, trade_signal_penalty, stored_charge):
   
    if discharge_signal:
        discharge_signal_penalty = stored_charge * trade_signal_penalty

        return discharge_signal_penalty


#### Penalise revenue by, price of, curtailed energy

In [15]:
def _interval_penalty_C(curtailment, curtailment_penalty):
    curtailed_energy_penalty = curtailment * curtailment_penalty
    
    return curtailed_energy_penalty

#### Penalise revenue by, import/export tarrifs

In [16]:
def _monthly_penalty_A(import_charge_peak, import_charge_shoulder, import_charge_off_peak, export_charge_sun_soaker, grid2battery_max_peak, grid2battery_max_shoulder, grid2battery_max_offpeak, export_max):
        
    demand_penalty = (import_charge_peak * grid2battery_max_peak) + (import_charge_shoulder * grid2battery_max_shoulder) + (import_charge_off_peak * grid2battery_max_offpeak) + (export_charge_sun_soaker * export_max)
    
    return demand_penalty

## Objective expression - interval level:

#### Each interval builds one symbolic revenue expression; the final objective sums them once.



In [17]:
def _interval_expression(price, grid_import_penalty, lgc_price, grid2battery, pv2grid, dischargeE_eff, import_charge, charge_signal, discharge_signal, charge_signal_penalty, discharge_signal_penalty, curtailed_energy_penalty):

    objective_components = {
        "exporting_energy_revenue": (pv2grid + dischargeE_eff) * price,
        "lgc_revenue": (pv2grid + dischargeE_eff - grid2battery * grid_import_penalty) * lgc_price,
        "importing_energy_cost": -(grid2battery * (price + import_charge)),
        "curtailed_energy_penalty": -curtailed_energy_penalty,
        "charge_signal_penalty": -charge_signal_penalty if charge_signal else 0,
        "discharge_signal_penalty": -discharge_signal_penalty if discharge_signal else 0,
    }

    interval_expression = lpSum(objective_components.values())

    return interval_expression, objective_components


## Objective expression - window level:

In [18]:
def _objective(self):
    
    self.Lp += lpSum(self.interval_objective_expressions) - lpSum(self.monthly_import_export_penalties), "Total_window_revenue"

    return self.Lp

## Within-window optimisation design logic:

### Interval level logic

In [19]:
def _interval_level_logic(self, i):

    # Dataset, quick reference
    generation = self.dataset_window_subset['Generation'].iloc[i]

    # Default signal and penalty values; overridden below when using BESS instructions
    charge_signal = False
    discharge_signal = False
    charge_signal_penalty = 0
    discharge_signal_penalty = 0

    predicted_price = self.dataset_window_subset["Predicted_price"].iloc[i]
    
    if variables.optimisation_directive == 2:
      
        # BESS instruction only
        if variables.predicted_price_type == 2:
            predicted_price = 0.0
            charge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 1
            discharge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 2

        # Both
        elif variables.predicted_price_type == 3:
            charge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 1
            discharge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 2

    lgc_price = self.dataset_window_subset["LGC"].iloc[i]
    import_charge = self.dataset_window_subset["Import_charge"][i]
    tou_label = self.dataset_window_subset["ToU_label"].iloc[i]
    gen_exportable = self.dataset_window_subset['Generation_exportable'][i]
    
    # Decision variables, quick reference
    pv2grid = self.pv2grid[i]
    pv2battery = self.pv2battery[i]
    curtailment = self.curtailment[i]
    grid2battery = self.grid2battery[i]
    dischargeE = self.dischargeE[i]
    stored_charge = self.stored_charge[i]

    year_month = self.window_year_month[i] 
    imports_max_peak = self.imports_max_peak[year_month]
    imports_max_shoulder = self.imports_max_shoulder[year_month]
    imports_max_offpeak = self.imports_max_offpeak[year_month]
    exports_max_sunsoaker = self.exports_max_sunsoaker[year_month]
    
    # Other calculated varibles, quick reference
    chargeE = self.pv2battery[i] + self.grid2battery[i]
    pv2battery_eff = pv2battery * self.dataset_window_subset["hte_from_pv"].iloc[i]
    grid2battery_eff = grid2battery * self.dataset_window_subset["hte_from_grid"].iloc[i]
    dischargeE_eff = dischargeE * self.dataset_window_subset["hte2grid"].iloc[i]

    def _previous_interval():
        previous_window_ending_mwh = self.window_start_soc * self.bess_mwh
        previous_interval_mwh = previous_window_ending_mwh if i == 0 else self.stored_charge[i - 1]
        return previous_interval_mwh
    
    previous_interval_mwh = _previous_interval()

    bess_max_eff = self.bess_mwh * self.dataset_window_subset["bess_deg"].iloc[i]
    bess_capacity_remaining = bess_max_eff - self.stored_charge[i]

    gen_exportable_now = self.dataset_window_subset['Generation_exportable'][i] > 0
    sun_soaker = self.dataset_window_subset["ToU_label"].iloc[i] != "Sun Soaker"

    
    # Call constraints
    self.Lp = _interval_constraint_A(self.Lp, generation, pv2grid, pv2battery, curtailment, i)
    self.constraint_map[f"Ballance_PV_output_{i}"] = i

    self.Lp = _interval_constraint_B(self.Lp, pv2battery_eff, grid2battery_eff, dischargeE, previous_interval_mwh, stored_charge, i)
    self.constraint_map[f"Ballance_BESS_charge_{i}"] = i

    self.Lp = _interval_constraint_C(self.Lp, pv2battery_eff, grid2battery_eff, dischargeE_eff, self.bess_energy_limit_per_interval_mw, i)
    self.constraint_map[f"Restrict_BESS_by_BESS_capacity_{i}"] = i
    
    self.Lp = _interval_constraint_D(self.Lp, stored_charge, bess_max_eff, chargeE, dischargeE, gen_exportable_now, pv2battery, gen_exportable, i)
    self.constraint_map[f"Restrict_stored_charge_by_physcial_limit_{i}"] = i
    self.constraint_map[f"Restrict_chargeE_by_physcial_limit_{i}"] = i
    self.constraint_map[f"Restrict_dischargeE_by_physcial_limit_{i}"] = i

    self.Lp = _interval_constraint_E(self.Lp, pv2grid, dischargeE_eff, variables.project_poi_export_limit_mwh_per_interval, grid2battery, variables.project_poi_import_limit_mwh_per_interval, i)
    self.constraint_map[f"Restrict_grid_by_grid_capacity_{i}"] = i

    if variables.optimisation_directive == 2 and variables.predicted_price_type != 1:
        self.Lp = _interval_constraint_F(self.Lp, charge_signal, discharge_signal, chargeE, dischargeE, i)
        self.constraint_map[f"Restrict_BESS_by_dischargeE_mutuality_{i}"] = i
        self.constraint_map[f"Restrict_BESS_by_chargeE_mutuality_{i}"] = i

    # self.Lp = _interval_constraint_G(self.Lp, sun_soaker, grid2battery, i)
    # self.constraint_map[f"Restrict_importing_by_sunsoaker_{i}"] = i

    self.Lp = _interval_constraint_H(self.Lp, tou_label, grid2battery, imports_max_peak, imports_max_shoulder, imports_max_offpeak, exports_max_sunsoaker, dischargeE_eff, pv2grid, i)
    self.constraint_map[f"Restrict_onpeak_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_shoulder_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_peak_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_sunsoaker_grid_exports_by_max_{i}"] = i

    # Call penalties
    if variables.optimisation_directive == 2 and variables.predicted_price_type != 1:
        charge_signal_penalty = _interval_penalty_A(bess_capacity_remaining, charge_signal, variables.trade_signal_penalty)
        discharge_signal_penalty = _interval_penalty_B(discharge_signal, variables.trade_signal_penalty, stored_charge)

    curtailed_energy_penalty = _interval_penalty_C(curtailment, variables.curtailment_penalty)

    # Call objective expression
    interval_expression, interval_objective_components = _interval_expression(predicted_price, variables.grid_import_penalty, lgc_price, grid2battery, pv2grid, dischargeE_eff, import_charge, charge_signal, discharge_signal, charge_signal_penalty, discharge_signal_penalty, curtailed_energy_penalty)
    for component_name, component_expr in interval_objective_components.items():
        self.objective_component_map.setdefault(component_name, {})[i] = component_expr
    self.interval_objective_expressions.append(interval_expression)

    return self


### Window level logic

In [20]:
def _window_level_logic(self):
    self.Lp = _window_constraint_A(self.Lp, self.window_cycle_budget_mwh, lpSum([self.dischargeE[i] for i in self.dataset_index_range]))

    return self

### Within-window controller

In [21]:
class Optimizer():

    def __init__(self, dataset_window_subset:pd.DataFrame, window_start_soc:float, dataset_index_range:range, bess_mwh, bess_energy_limit_per_interval_mw):
        self.dataset_window_subset = dataset_window_subset
        self.window_start_soc = window_start_soc
        self.dataset_index_range = dataset_index_range
        self.interval_objective_expressions = []
        self.monthly_import_export_penalties = []
        self.objective_component_map = {}
        self.constraint_map = {}

        self.bess_mwh = bess_mwh
        self.bess_energy_limit_per_interval_mw = bess_energy_limit_per_interval_mw

        self.window_months, self.window_year_month = self._calculate_years_months()
        self.window_cycle_budget_mwh = self._calculate_cycle_budget()
        self._main_loop()
    

    def _calculate_years_months(self):
        window_dates = pd.to_datetime(self.dataset_window_subset["Date"])
        window_year_month = list(zip(window_dates.dt.year.to_numpy(), window_dates.dt.month.to_numpy()))
        window_months = sorted(set(window_year_month))
        return window_months, window_year_month
        
    def _calculate_cycle_budget(self):
        periods_per_day = 24 * (60 / variables.operation_granularity_in_minutes)
        days_in_window = len(self.dataset_window_subset) / periods_per_day
        avg_window_deg = np.mean(self.dataset_window_subset["bess_deg"])
        window_cycle_budget_mwh = variables.cycles_per_day * self.bess_mwh * avg_window_deg * days_in_window
        return window_cycle_budget_mwh
    
    def _main_loop(self):
        # Define Linear Programming optimization type
        self.Lp = LpProblem("Optimization", LpMaximize)
        
        # Define decision variables
        self.stored_charge = [LpVariable(f"stored_charge_{i}", lowBound=0) for i in self.dataset_index_range]
        self.pv2grid = [LpVariable(f"pv2grid_{i}", lowBound=0) for i in self.dataset_index_range]
        self.pv2battery = [LpVariable(f"pv2battery_{i}", lowBound=0) for i in self.dataset_index_range]
        self.curtailment = [LpVariable(f"curtailment_{i}", lowBound=0) for i in self.dataset_index_range]
        self.dischargeE = [LpVariable(f"dischargeE_{i}", lowBound=0) for i in self.dataset_index_range]
        self.grid2battery = [LpVariable(f"grid2battery_{i}", lowBound=0, upBound=variables.project_poi_import_limit_mwh_per_interval,) for i in self.dataset_index_range]
        self.imports_max_peak = {ym: LpVariable(f"imports_max_peak_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.imports_max_shoulder = {ym: LpVariable(f"imports_max_shoulder_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.imports_max_offpeak = {ym: LpVariable(f"imports_max_offpeak_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.exports_max_sunsoaker = {ym: LpVariable(f"exports_max_sunsoaker_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
     
        # Create per month objective penalties here
        for ym in self.window_months:

            monthly_import_export_penalty = _monthly_penalty_A(variables.demand_charge_peak,variables.demand_charge_shoulder,variables.demand_charge_off_peak,variables.export_charge_sun_soaker,self.imports_max_peak[ym],self.imports_max_shoulder[ym],self.imports_max_offpeak[ym],self.exports_max_sunsoaker[ym])
            self.monthly_import_export_penalties.append(monthly_import_export_penalty)
            first_interval_of_month = self.window_year_month.index(ym)
            self.objective_component_map.setdefault("monthly_import_export_penalties", {})[first_interval_of_month] = -monthly_import_export_penalty

        # Create per INTERVAL constraints and objectives
        for i in self.dataset_index_range:
            _interval_level_logic(self, i)

        # Create per WINDOW constraints and objectives
        _window_level_logic(self)

        # Create objective
        self.Lp = _objective(self)

        # Solve
        self.model = self.Lp.solve(PULP_CBC_CMD(msg=0))


## Window scheduler logic:

In [22]:
def create_optimisation(inputs, item=None):

    # get the data the normal way, through the passed dataset
    def get_inputs_single_series():
        dataset = inputs["dataset"]
        bess_mwh = variables.bess_mwh
        bess_energy_limit_per_interval_mw = variables.bess_energy_limit_per_interval_mw

        return dataset, bess_mwh, bess_energy_limit_per_interval_mw
    
    # modify the dataset with new item conbination data based on this passed item
    def get_inputs_multi_series():
        dataset = inputs["dataset"]
        multi_series_actual_price = inputs["multi_series_actual_price"]
        multi_series_predicted_price = inputs["multi_series_predicted_price"]
        multi_series_bess = inputs["multi_series_bess"]
        price_col_indexer, bess_col_indexer = item

        dataset["Actual_price"] = multi_series_actual_price[price_col_indexer]
        dataset["Predicted_price"] = multi_series_predicted_price[price_col_indexer]

        bess_mwh = multi_series_bess.iloc[bess_col_indexer]["BESS_MWh"]
        bess_energy_limit_per_interval_mw = multi_series_bess.iloc[bess_col_indexer]["BESS_MWh_per_interval"]

        bess_row = multi_series_bess.iloc[bess_col_indexer]

        return dataset, bess_mwh, bess_energy_limit_per_interval_mw, price_col_indexer, bess_row

    if variables.run_multiple_optimisations == False:
        dataset, bess_mwh, bess_energy_limit_per_interval_mw = get_inputs_single_series()
    else:
        dataset, bess_mwh, bess_energy_limit_per_interval_mw, price_col_indexer, bess_row = get_inputs_multi_series()


    length_of_single_lookahead_window = int(int(variables.optimization_avoid_edge_effect_total_hours) * (60 / int(variables.operation_granularity_in_minutes)))
    length_of_window_minus_lookahead = int(int(variables.number_of_intervals_per_window) - length_of_single_lookahead_window)

    retained_outputs = []
    window_start_soc = variables.start_soc
    window_interval_stride = length_of_window_minus_lookahead

    if variables.display_window_scheduler_visual and not variables.run_multiple_optimisations:
        # Visualisation purposes only
        window_starts = list(range(0, len(dataset), window_interval_stride))
        intervals_per_bar = length_of_single_lookahead_window
        bars_per_window = int(variables.number_of_intervals_per_window / intervals_per_bar)
        retained_bars_per_window = int(length_of_window_minus_lookahead / intervals_per_bar)
        fig, completed_x, completed_y, completed_colors = create_window_progress_visual(
            window_starts=window_starts,
            total_dataset_intervals=len(dataset),
            intervals_per_bar=intervals_per_bar,
            bars_per_window=bars_per_window,
            retained_bars_per_window=retained_bars_per_window,
            dataset=dataset,
        )
        
    for i, start_interval in enumerate(range(0, len(dataset), window_interval_stride)):

        stop_interval = min(start_interval + variables.number_of_intervals_per_window, len(dataset))
        dataset_window_subset = dataset.iloc[start_interval:stop_interval].copy().reset_index(drop=True)
        dataset_index_range = range(len(dataset_window_subset))
        
        Optimization = Optimizer(dataset_window_subset=dataset_window_subset, window_start_soc=window_start_soc, dataset_index_range=dataset_index_range, bess_mwh=bess_mwh, bess_energy_limit_per_interval_mw=bess_energy_limit_per_interval_mw)

        optimization_output = post_process_window_logic(Optimization)
        
        optimization_output["Window"] = i

        if stop_interval == len(dataset):
            retained_optimization_output = optimization_output
        else:
            retained_optimization_output = optimization_output.iloc[:length_of_window_minus_lookahead].copy()
            last_retained_stored_charge_mwh = optimization_output["interval_stored_charge"].iloc[length_of_window_minus_lookahead - 1]
            window_start_soc = last_retained_stored_charge_mwh / bess_mwh

        retained_outputs.append(retained_optimization_output)

        if variables.display_window_scheduler_visual and not variables.run_multiple_optimisations:
            # Visualisation purposes only
            actual_bars_this_window = min(bars_per_window, int((stop_interval - start_interval) / intervals_per_bar))
            if actual_bars_this_window > 0:
                update_window_progress_visual(fig=fig, current_window=i, current_horizon_bar=actual_bars_this_window - 1, window_starts=window_starts, bars_per_window=bars_per_window, completed_x=completed_x, completed_y=completed_y, completed_colors=completed_colors, force=(stop_interval == len(dataset)))

    retained_outputs = pd.concat(retained_outputs, ignore_index=True, sort=False)
    retained_outputs = retained_outputs.round(4)
    
    # Call the export function
    export_df = export_outut_to_csv(retained_outputs, price_col_indexer, bess_row["BESS_Duration_h"])

    return export_df


## Prepare optimization inputs (Code entry point)

In [23]:
def prepare_inputs():

    row_limit = 8760 * 30
    price_series_limit = 1
    bess_series_limit = 1

    # dataset - core dataset contain data to optimizer over
    dataset = pd.read_csv("../1_Dataset/2_Processed_data/Dataset/dataset.csv")
    dataset["Generation_exportable"] = np.clip(
        dataset["Generation"] - variables.project_poi_export_limit_mwh_per_interval,
        a_min=0,a_max=None
    )
    dataset = dataset[:row_limit]

    
    multi_series_price_actual = pd.read_csv("../1_Dataset/2_Processed_data/resampled_aurora_price.csv")
    operation_start_date = pd.to_datetime(variables.operation_start_date, format="%d/%m/%Y %H:%M")
    operation_end_date = pd.to_datetime(variables.operation_end_date, format="%d/%m/%Y %H:%M")
    multi_series_price_actual["Date"] = pd.to_datetime(multi_series_price_actual["Date"])
    multi_series_price_actual = multi_series_price_actual.loc[(multi_series_price_actual["Date"] >= operation_start_date)& (multi_series_price_actual["Date"] <= operation_end_date)].copy()
    multi_series_price_actual = multi_series_price_actual.drop(columns=["Date"]).reset_index(drop=True)
    
    if variables.optimisation_directive == 1:
        multi_series_actual_price = multi_series_price_actual.iloc[:row_limit,:price_series_limit]
        multi_series_predicted_price = multi_series_price_actual.iloc[:row_limit,:price_series_limit]
    else:
        multi_series_price_predicted = pd.read_csv("../1_Dataset/2_Processed_data/predicted_prices.csv")
        multi_series_price_predicted = multi_series_price_predicted.drop(columns=["Date"])

        multi_series_actual_price = multi_series_price_actual.iloc[:row_limit,:price_series_limit]
        multi_series_predicted_price = multi_series_price_predicted.iloc[:row_limit,:price_series_limit]
    
    # multi_series_bess
    multi_series_bess = pd.read_csv("../1_Dataset/2_Processed_data/bess_designs.csv")
    multi_series_bess = multi_series_bess.iloc[:bess_series_limit]

    return {
        "dataset": dataset,
        "multi_series_actual_price": multi_series_actual_price,
        "multi_series_predicted_price": multi_series_predicted_price,
        "multi_series_bess": multi_series_bess,
    }

inputs = prepare_inputs()

dataset = inputs["dataset"]
multi_series_actual_price = inputs["multi_series_actual_price"]
multi_series_predicted_price = inputs["multi_series_predicted_price"]
multi_series_bess = inputs["multi_series_bess"]

FileNotFoundError: [Errno 2] No such file or directory: '../1_Dataset/2_Processed_data/predicted_prices.csv'

In [ ]:
dataset[:5]

,Date,Generation,LGC,Import_charge,ToU_label,bess_deg,hte_from_pv,hte_from_grid,hte2grid,pv_bess_grid,grid_bess_grid,Generation_exportable
0,2026-01-01 00:00:00,0.0,6.48,0.0,Off-peak,1.000000,0.926727,0.880467,0.942151,0.851046,0.829534,0.0
1,2026-01-01 01:00:00,0.0,6.48,0.0,Off-peak,0.999995,0.926726,0.880467,0.942151,0.851046,0.829533,0.0
2,2026-01-01 02:00:00,0.0,6.48,0.0,Off-peak,0.999991,0.926726,0.880467,0.942151,0.851046,0.829533,0.0
3,2026-01-01 03:00:00,0.0,6.48,0.0,Off-peak,0.999986,0.926726,0.880466,0.942151,0.851045,0.829533,0.0
4,2026-01-01 04:00:00,0.0,6.48,0.0,Off-peak,0.999982,0.926725,0.880466,0.942151,0.851045,0.829532,0.0


In [ ]:
multi_series_actual_price[:5]

,Sample 1
0,116.17
1,35.58
2,34.66
3,121.44
4,20.28


In [ ]:
multi_series_predicted_price[:5]

,Sample 1
0,116.17
1,35.58
2,34.66
3,121.44
4,20.28


In [ ]:
multi_series_bess[:5]

,BESS_Duration_h,BESS_Power_MW,BESS_MWh,BESS_MWh_per_interval
0,2.0,4.96,9.382,4.96


## Singular / parrallel coordinator

In [ ]:
# single series operating mode
if variables.run_multiple_optimisations == False:
    results = create_optimisation(inputs)
    print(results["actual_revenue_row_total"].sum())

# multi series operating mode
if variables.run_multiple_optimisations == True:

    def create_dataset_combinations():
        
        # Assign combinations of items
        items = [
            (price_col_indexer, bess_col_indexer)
            for price_col_indexer in multi_series_predicted_price.columns
            for bess_col_indexer in range(len(multi_series_bess))
        ]

        return items
    
    if variables.execution_mode == "sequential":   
        function = partial(create_optimisation, inputs)

        results = []

        for item in tqdm(create_dataset_combinations()):
            results.append(function(item))

        results = pd.concat(results, ignore_index=True)

    if variables.execution_mode == "parallel":

        data=create_dataset_combinations()
        function = partial(create_optimisation, inputs)

        results = pd.concat(run_parallel(
            function=function,
            data=data
        ), ignore_index=True)


results.to_csv("Optimization_data/optimization_results.csv", index=False)


100%|██████████| 1/1 [00:25<00:00, 25.59s/it]
